In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine

from src.instruments.deposit_instrument import DepositInstrument

In [2]:
# downloading market curves
loader = MarketLoader()

curves = loader.loader_pipeline()
curves

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..


treasury                                            sofr           \
                 1M    3M    6M    1Y    2Y    5Y   10Y   30Y    ON FEDFUNDS   
Date                                                                           
2019-10-01     1.79  1.82  1.81  1.73  1.56  1.51  1.65  2.11  1.88     1.88   
2019-10-02     1.75  1.79  1.75  1.67  1.48  1.43  1.60  2.09  1.85     1.85   
2019-10-03     1.78  1.70  1.66  1.58  1.39  1.34  1.54  2.04  1.84     1.83   
2019-10-04     1.73  1.71  1.65  1.58  1.40  1.34  1.52  2.01  1.82     1.82   
2019-10-07     1.76  1.75  1.73  1.64  1.46  1.38  1.56  2.05  1.83     1.82   
...             ...   ...   ...   ...   ...   ...   ...   ...   ...      ...   
2026-05-22     3.72  3.68  3.79  3.86  4.13  4.27  4.56  5.07  3.55     3.62   
2026-05-25     3.72  3.68  3.79  3.86  4.13  4.27  4.56  5.07  3.55     3.62   
2026-05-26     3.72  3.68  3.80  3.82  4.01  4.19  4.50  5.03  3.63     3.62   
2026-05-27     3.72  3.68  3.79  3.80  4.00  4.17  4.48  5.01  3.63     3.62   
2026-05-28     3.72  3.68  3.79  3.80  4.00  4.17  4.48  5.01  3.63     3.62   

           futures           estr  
           TBill3M TBill6M   ESTR  
Date                               
2019-10-01    1.78    1.76 -0.549  
2019-10-02    1.75    1.71 -0.551  
2019-10-03    1.67    1.62 -0.555  
2019-10-04    1.68    1.61 -0.553  
2019-10-07    1.71    1.69 -0.554  
...            ...     ...    ...  
2026-05-22    3.59    3.64  1.929  
2026-05-25    3.59    3.64  1.931  
2026-05-26    3.60    3.65  1.932  
2026-05-27    3.59    3.64  1.932  
2026-05-28    3.59    3.64  1.933  

[1738 rows x 13 columns]

In [3]:
# curve snapshot of a given date
treasury_df = curves['treasury']

latest_date = treasury_df.index[-1]
latest_curve = treasury_df.iloc[-1]

snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'treasury',
    as_of_date = latest_date,
    curve_row = latest_curve
)

snapshot.summary()

,Tenor,Years,Rate
0,1M,0.083333,3.72
1,3M,0.250000,3.68
2,6M,0.500000,3.79
3,1Y,1.000000,3.80
4,2Y,2.000000,4.00
5,5Y,5.000000,4.17
6,10Y,10.000000,4.48
7,30Y,30.000000,5.01


In [5]:
# create deposit instruments from treasury snapshot
deposit_instruments = []

for tenor, rate in zip(snapshot.tenors, snapshot.rates):

    if tenor in ('1M', '3M', '6M', '1Y', '2Y', '3Y'):
        deposit_instruments.append(
            DepositInstrument(
                tenor = tenor,
                market_rate = rate
            )
        )

display(deposit_instruments)

# bootstrap discount curve
engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = snapshot,
    instruments = deposit_instruments
)

discount_curve.summary()

[DepositInstrument(type=deposit, tenor=1M, rate=3.72),
 DepositInstrument(type=deposit, tenor=3M, rate=3.68),
 DepositInstrument(type=deposit, tenor=6M, rate=3.79),
 DepositInstrument(type=deposit, tenor=1Y, rate=3.8),
 DepositInstrument(type=deposit, tenor=2Y, rate=4.0)]

,Maturity,DiscountFactor
0,0.083333,0.996910
1,0.250000,0.990884
2,0.500000,0.981402
3,1.000000,0.963391
4,2.000000,0.925926
